<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 12


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----
Создать базовый класс Item в C#, который будет представлять информацию о
товарах, которые могут быть заказаны или возвращены. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public delegate void ItemPriceChangedHandler(int itemId, string itemName, decimal oldPrice, decimal newPrice);

public class Item
{
    public int ItemId { get; set; }
    public string Name { get; set; }
    public decimal Price { get; set; }
    public string Category { get; set; }
    public int StockQuantity { get; set; }
    public bool IsReturnable { get; set; }
    public double WeightKg { get; set; }

    public event ItemPriceChangedHandler PriceChanged;

    public Item(int itemId, string name, decimal price, string category, int stockQuantity, bool isReturnable, double weightKg)
    {
        ItemId = itemId;
        Name = name;
        Price = price;
        Category = category;
        StockQuantity = stockQuantity;
        IsReturnable = isReturnable;
        WeightKg = weightKg;
    }

    protected void OnPriceChanged(decimal oldPrice, decimal newPrice)
    {
        PriceChanged?.Invoke(ItemId, Name, oldPrice, newPrice);
    }

    public virtual string GetDetails()
    {
        return $"ID: {ItemId}, Товар: {Name}, Цена: {Price:F2}, Категория: {Category}, Остаток: {StockQuantity}";
    }

    public virtual decimal CalculateDiscount()
    {
        return Price * 0.05m;
    }

    public virtual void ApplyDiscount(decimal discount)
    {
        decimal oldPrice = Price;
        if (discount > 0 && discount < Price)
        {
            Price -= discount;
            OnPriceChanged(oldPrice, Price);
        }
    }

    public void Restock(int amount)
    {
        if (amount > 0)
        {
            StockQuantity += amount;
        }
    }

    public void MarkAsNonReturnable()
    {
        IsReturnable = false;
    }

    public double CalculateTotalWeight(int count)
    {
        return WeightKg * count;
    }
}

public class SingleItem : Item
{
    public string UnitMeasure { get; set; }
    public string Manufacturer { get; set; }
    public string Barcode { get; set; }
    public bool RequiresSerialNumber { get; set; }

    public SingleItem(int itemId, string name, decimal price, string category, int stockQuantity, bool isReturnable, double weightKg,
                      string unitMeasure, string manufacturer, string barcode, bool requiresSerialNumber)
        : base(itemId, name, price, category, stockQuantity, isReturnable, weightKg)
    {
        UnitMeasure = unitMeasure;
        Manufacturer = manufacturer;
        Barcode = barcode;
        RequiresSerialNumber = requiresSerialNumber;
    }

    public override string GetDetails()
    {
        return $"{base.GetDetails()}, Ед. изм.: {UnitMeasure}, Производитель: {Manufacturer}";
    }

    public void UpdateBarcode(string newBarcode)
    {
        Barcode = newBarcode;
    }

    public string FormatLabel()
    {
        return $"[{Barcode}] {Name} ({UnitMeasure}) — {Price:F2}";
    }

    public bool CheckSerialRequirement()
    {
        return RequiresSerialNumber;
    }
}

public class PackageItem : Item
{
    public int QuantityPerPackage { get; set; }
    public string PackageType { get; set; }
    public decimal PackageDiscountRate { get; set; }
    public bool AllowPartialUnpack { get; set; }

    public PackageItem(int itemId, string name, decimal price, string category, int stockQuantity, bool isReturnable, double weightKg,
                       int quantityPerPackage, string packageType, decimal packageDiscountRate, bool allowPartialUnpack)
        : base(itemId, name, price, category, stockQuantity, isReturnable, weightKg)
    {
        QuantityPerPackage = quantityPerPackage;
        PackageType = packageType;
        PackageDiscountRate = packageDiscountRate;
        AllowPartialUnpack = allowPartialUnpack;
    }

    public override decimal CalculateDiscount()
    {
        return Price * QuantityPerPackage * PackageDiscountRate;
    }

    public decimal CalculateUnitPrice()
    {
        return QuantityPerPackage > 0 ? Price / QuantityPerPackage : Price;
    }

    public void ChangePackageType(string newType)
    {
        PackageType = newType;
    }

    public int TotalUnitsInStock()
    {
        return StockQuantity * QuantityPerPackage;
    }
}

public class SpecialItem : Item
{
    public DateTime DiscountExpirationDate { get; set; }
    public string PromoCode { get; set; }
    public int MaxPerOrder { get; set; }
    public bool IsLimitedEdition { get; set; }

    public SpecialItem(int itemId, string name, decimal price, string category, int stockQuantity, bool isReturnable, double weightKg,
                       DateTime discountExpirationDate, string promoCode, int maxPerOrder, bool isLimitedEdition)
        : base(itemId, name, price, category, stockQuantity, isReturnable, weightKg)
    {
        DiscountExpirationDate = discountExpirationDate;
        PromoCode = promoCode;
        MaxPerOrder = maxPerOrder;
        IsLimitedEdition = isLimitedEdition;
    }

    public override void ApplyDiscount(decimal discount)
    {
        decimal oldPrice = Price;
        if (DateTime.Now <= DiscountExpirationDate && discount > 0 && discount < Price)
        {
            Price -= discount;
            OnPriceChanged(oldPrice, Price);
        }
        Console.WriteLine($"Срок действия скидки ({PromoCode}) истекает: {DiscountExpirationDate:dd.MM.yyyy}");
    }

    public bool IsDiscountActive()
    {
        return DateTime.Now <= DiscountExpirationDate;
    }

    public void ExtendDiscount(int extraDays)
    {
        DiscountExpirationDate = DiscountExpirationDate.AddDays(extraDays);
    }

    public void SetOrderLimit(int limit)
    {
        if (limit > 0)
        {
            MaxPerOrder = limit;
        }
    }
}

List<Item> catalog = new List<Item>
{
    new SingleItem(1, "Кабель HDMI", 1200m, "Электроника", 15, true, 0.2, "шт.", "Ugreen", "4601234567890", false),
    new PackageItem(2, "Набор фильтров", 4500m, "Автотовары", 8, true, 1.5, 6, "Коробка", 0.10m, false),
    new SpecialItem(3, "Лимитированный комплект", 10000m, "Акции", 5, false, 2.0, DateTime.Now.AddDays(7), "PROMO2026", 2, true)
};

Dictionary<int, Item> itemIndex = new Dictionary<int, Item>();
HashSet<string> categories = new HashSet<string>();

Func<Item, bool> isExpensive = i => i.Price >= 3000m;
Action<string> printLine = msg => Console.WriteLine(msg);

foreach (Item item in catalog)
{
    item.PriceChanged += (id, name, oldP, newP) =>
        printLine($"[Событие] Цена '{name}' (ID {id}) изменена: {oldP:F2} -> {newP:F2}");

    itemIndex[item.ItemId] = item;
    categories.Add(item.Category);

    printLine(item.GetDetails());
    printLine($"Расчетная скидка: {item.CalculateDiscount():F2}, Дорогой товар: {isExpensive(item)}");
    item.ApplyDiscount(500m);
    printLine("");
}

printLine($"Уникальных категорий в каталоге: {categories.Count}");
if (itemIndex.TryGetValue(2, out Item found))
{
    printLine($"Поиск по ID 2: {found.Name}");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

ID: 1, Товар: Кабель HDMI, Цена: 1200,00, Категория: Электроника, Остаток: 15, Ед. изм.: шт., Производитель: Ugreen
Расчетная скидка: 60,00, Дорогой товар: False
[Событие] Цена 'Кабель HDMI' (ID 1) изменена: 1200,00 -> 700,00

ID: 2, Товар: Набор фильтров, Цена: 4500,00, Категория: Автотовары, Остаток: 8
Расчетная скидка: 2700,00, Дорогой товар: True
[Событие] Цена 'Набор фильтров' (ID 2) изменена: 4500,00 -> 4000,00

ID: 3, Товар: Лимитированный комплект, Цена: 10000,00, Категория: Акции, Остаток: 5
Расчетная скидка: 500,00, Дорогой товар: True
[Событие] Цена 'Лимитированный комплект' (ID 3) изменена: 10000,00 -> 9500,00
Срок действия скидки (PROMO2026) истекает: 26.11.2026

Уникальных категорий в каталоге: 3
Поиск по ID 2: Набор фильтров
